# Strava Tell Me More 🚴‍♂️

Generate engaging historical stories about your real cycling routes using Strava API and OpenAI.

## Features
- 🚴‍♂️ Real Strava API integration with token management
- 🤖 AI-powered storytelling with multiple themes
- 🔒 Secure environment variable configuration
- 📊 Smart fallback to mock data when needed
- 🔧 Automatic token refresh and error handling

In [3]:
# Import required libraries
import pandas as pd
import os
import openai
from typing import List, Dict, Optional
import json
from datetime import datetime
import textwrap
from dotenv import load_dotenv
import requests

# Load environment variables from .env file
load_dotenv()
print("📦 Libraries imported successfully!")

📦 Libraries imported successfully!


In [4]:
# Configuration - API keys from environment variables
openai_key = os.getenv('OPENAI_API_KEY')
strava_client_id = os.getenv('STRAVA_CLIENT_ID')
strava_client_secret = os.getenv('STRAVA_CLIENT_SECRET')
strava_access_token = os.getenv('STRAVA_ACCESS_TOKEN')
strava_refresh_token = os.getenv('STRAVA_REFRESH_TOKEN')

if not openai_key:
    raise ValueError("OPENAI_API_KEY environment variable not set. Please check your .env file.")

chat_model = "gpt-3.5-turbo"
print("🔑 Configuration loaded from environment variables")

🔑 Configuration loaded from environment variables


## 🔧 Fix Permission Errors

If you're getting **"activity:read_permission missing"** errors, your Strava app doesn't have the right permissions.

### Quick Fix:
1. **Check your Client ID** in your `.env` file
2. **Visit this URL** (replace `YOUR_CLIENT_ID` with your actual Client ID):
   ```
   https://www.strava.com/oauth/authorize?client_id=YOUR_CLIENT_ID&response_type=code&redirect_uri=http://localhost&approval_prompt=force&scope=read,activity:read
   ```
3. **Authorize the app** with the correct scopes: `read,activity:read`
4. **Get the authorization code** from the redirect URL
5. **Exchange it for access/refresh tokens** using Strava's token endpoint

### The Problem:
Your current tokens were likely created without the `activity:read` scope, so they can only access public profile information, not your activities.

### The Solution:
Re-authorize your app with the proper scopes to get new tokens that can read your cycling activities.

In [5]:
# Strava Token Management Class
class StravaTokenManager:
    """Handles Strava token validation, refresh, and management."""
    
    def __init__(self, client_id, client_secret, access_token, refresh_token=None):
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = access_token
        self.refresh_token = refresh_token
        self.token_expires_at = None
        
    def validate_token(self):
        """Check if the current access token is valid."""
        if not self.access_token:
            return False, "No access token provided"
            
        # Test token by making a simple API call
        headers = {"Authorization": f"Bearer {self.access_token}"}
        try:
            response = requests.get("https://www.strava.com/api/v3/athlete", headers=headers, timeout=10)
            
            if response.status_code == 200:
                return True, "Token is valid"
            elif response.status_code == 401:
                return False, "Token is expired or invalid"
            else:
                return False, f"API returned status {response.status_code}"
                
        except requests.RequestException as e:
            return False, f"Network error during token validation: {e}"
    
    def refresh_access_token(self):
        """Refresh the access token using the refresh token."""
        if not self.refresh_token or not self.client_id or not self.client_secret:
            return False, "Missing refresh token or client credentials"
        
        refresh_url = "https://www.strava.com/oauth/token"
        refresh_data = {
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "refresh_token": self.refresh_token,
            "grant_type": "refresh_token"
        }
        
        try:
            response = requests.post(refresh_url, data=refresh_data, timeout=10)
            
            if response.status_code == 200:
                token_data = response.json()
                self.access_token = token_data.get("access_token")
                self.refresh_token = token_data.get("refresh_token")
                expires_at = token_data.get("expires_at")
                
                if expires_at:
                    self.token_expires_at = datetime.fromtimestamp(expires_at)
                
                return True, "Token refreshed successfully"
            else:
                error_data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {"error": "Unknown error"}
                return False, f"Token refresh failed: {error_data.get('message', 'Unknown error')}"
                
        except Exception as e:
            return False, f"Error during token refresh: {e}"
    
    def get_valid_token(self):
        """Get a valid access token, refreshing if necessary."""
        # First, check if current token is valid
        is_valid, message = self.validate_token()
        
        if is_valid:
            return True, self.access_token, "Current token is valid"
        
        # If not valid, try to refresh
        print(f"🔄 Token validation failed: {message}")
        print("🔄 Attempting to refresh token...")
        
        refresh_success, refresh_message = self.refresh_access_token()
        
        if refresh_success:
            print(f"✅ {refresh_message}")
            return True, self.access_token, refresh_message
        else:
            print(f"❌ Token refresh failed: {refresh_message}")
            return False, None, refresh_message

print("✅ StravaTokenManager class defined")

✅ StravaTokenManager class defined


In [6]:
# Strava API Setup with Token Management
try:
    from stravalib import Client
    from stravalib.exc import Fault as StravaFault
    
    # Initialize Strava client with proper token management
    strava_client = None
    strava_token_manager = None
    
    if strava_access_token or strava_refresh_token:
        print("🔍 Initializing Strava token management...")
        
        # Create token manager
        strava_token_manager = StravaTokenManager(
            client_id=strava_client_id,
            client_secret=strava_client_secret,
            access_token=strava_access_token,
            refresh_token=strava_refresh_token
        )
        
        # Get a valid token
        token_valid, valid_token, token_message = strava_token_manager.get_valid_token()
        
        if token_valid and valid_token:
            try:
                # Create client with both access and refresh tokens
                strava_client = Client()
                strava_client.access_token = valid_token
                
                # Set refresh token if available to suppress warnings
                if strava_refresh_token:
                    strava_client.refresh_token = strava_refresh_token
                    strava_client.token_expires_at = strava_token_manager.token_expires_at
                
                # Test the client with a simple API call
                try:
                    athlete = strava_client.get_athlete()
                    print(f"✅ Strava API client initialized successfully")
                    print(f"👤 Connected as: {athlete.firstname} {athlete.lastname}")
                    
                    # Check if we have activity read permissions
                    try:
                        # Try to get a single activity to test permissions
                        activities_test = list(strava_client.get_activities(limit=1))
                        print("✅ Activity read permissions confirmed")
                    except Exception as perm_error:
                        if 'activity:read_permission' in str(perm_error) or 'missing' in str(perm_error).lower():
                            print("❌ Missing activity:read permission")
                            print("💡 Your Strava app needs 'activity:read' scope")
                            print("📋 Follow the instructions in the cell above to reauthorize")
                            strava_client = None  # Don't use client without proper permissions
                        else:
                            print(f"⚠️  Permission test inconclusive: {perm_error}")
                    
                except Exception as api_error:
                    if 'activity:read_permission' in str(api_error) or 'missing' in str(api_error).lower():
                        print("❌ Missing activity:read permission")
                        print("💡 Your Strava app needs 'activity:read' scope")
                        print("📋 Follow the instructions in the cell above to reauthorize")
                        strava_client = None
                    else:
                        print(f"⚠️  Client created but API test failed: {api_error}")
                        print("🔄 Will attempt to use client anyway...")
                    
            except Exception as client_error:
                print(f"❌ Error creating Strava client: {client_error}")
                strava_client = None
        else:
            print(f"❌ Could not obtain valid token: {token_message}")
            print("📋 Please check your Strava credentials in the .env file")
            
    else:
        print("⚠️  No Strava access token or refresh token found")
        print("📋 Add STRAVA_ACCESS_TOKEN or STRAVA_REFRESH_TOKEN to your .env file")
        print("🔄 Will use mock data for demonstration")
        
except ImportError:
    print("⚠️  stravalib not installed")
    print("💡 Install with: pip install stravalib")
    print("🔄 Will use mock data for demonstration")
    strava_client = None
    strava_token_manager = None
    
except Exception as e:
    print(f"❌ Unexpected error during Strava setup: {e}")
    strava_client = None
    strava_token_manager = None

# Display final status
if strava_client:
    print("🟢 Strava Status: Connected and ready")
elif strava_token_manager:
    print("🟡 Strava Status: Token manager ready, but client creation failed")
else:
    print("🔴 Strava Status: Not available - will use mock data")

🔍 Initializing Strava token management...


✅ Strava API client initialized successfully
👤 Connected as: Luke John Roberts
❌ Missing activity:read permission
💡 Your Strava app needs 'activity:read' scope
📋 Follow the instructions in the cell above to reauthorize
🟡 Strava Status: Token manager ready, but client creation failed


In [7]:
class StravaAPI:
    """Handles Strava API interactions with robust error handling and token management."""
    
    def __init__(self, client=None, token_manager=None):
        self.client = client
        self.token_manager = token_manager
        self.mock_locations = ["Segbroek", "Kraayenstein", "Kwintsheul", "Schipluiden", 
                              "Berkel en Rodenrijs", "Pijnacker", "Leidschendam", "Marlot", "Haagse Bos"]
    
    def _refresh_client_if_needed(self):
        """Refresh the client with a new token if needed."""
        if not self.token_manager:
            return False
        
        try:
            token_valid, valid_token, message = self.token_manager.get_valid_token()
            if token_valid and valid_token:
                self.client = Client()
                self.client.access_token = valid_token
                if self.token_manager.refresh_token:
                    self.client.refresh_token = self.token_manager.refresh_token
                print("🔄 Client refreshed with new token")
                return True
            return False
        except Exception as e:
            print(f"❌ Error refreshing client: {e}")
            return False
    
    def _make_api_call_with_retry(self, api_call, *args, **kwargs):
        """Make an API call with automatic retry on token expiration."""
        max_retries = 2
        
        for attempt in range(max_retries):
            try:
                if not self.client:
                    raise Exception("No Strava client available")
                
                return api_call(*args, **kwargs)
                
            except Exception as e:
                error_str = str(e).lower()
                
                # Check if it's a token-related error
                if any(token_error in error_str for token_error in ['unauthorized', 'invalid', 'expired', '401']):
                    if attempt < max_retries - 1:  # Not the last attempt
                        print(f"🔄 Token error detected (attempt {attempt + 1}): {e}")
                        if self._refresh_client_if_needed():
                            continue  # Retry with new token
                        else:
                            break  # Can't refresh, give up
                    else:
                        print(f"❌ Token error persists after retries: {e}")
                        raise e
                else:
                    # Non-token error, don't retry
                    raise e
        
        raise Exception("API call failed after retries")
    
    def get_recent_activities(self, limit=10):
        """Fetch recent cycling activities from Strava with retry logic."""
        if not self.client:
            print("🔄 No Strava client available, using mock data")
            return self._get_mock_activities()
        
        try:
            activities = list(self._make_api_call_with_retry(
                self.client.get_activities, 
                limit=limit
            ))
            
            cycling_activities = [a for a in activities if a.type.lower() in ['ride', 'cycling']]
            
            if not cycling_activities:
                print("🔄 No cycling activities found, using mock data")
                return self._get_mock_activities()
                
            print(f"✅ Found {len(cycling_activities)} cycling activities")
            return cycling_activities
            
        except Exception as e:
            print(f"❌ Error fetching activities: {e}")
            print("🔄 Falling back to mock data")
            return self._get_mock_activities()
    
    def get_activity_details(self, activity_id):
        """Fetch detailed activity data with retry logic."""
        if not self.client:
            return None
            
        try:
            return self._make_api_call_with_retry(
                self.client.get_activity, 
                activity_id
            )
        except Exception as e:
            print(f"❌ Error fetching activity details for {activity_id}: {e}")
            return None
    
    def extract_route_coordinates(self, activity):
        """Extract GPS coordinates from activity streams with retry logic."""
        if not self.client or not activity:
            return []
            
        try:
            streams = self._make_api_call_with_retry(
                self.client.get_activity_streams,
                activity.id, 
                types=['latlng'], 
                resolution='medium'
            )
            
            if 'latlng' in streams:
                coordinates = streams['latlng'].data
                sampled_coords = coordinates[::10] if len(coordinates) > 20 else coordinates
                print(f"✅ Extracted {len(sampled_coords)} GPS coordinates")
                return sampled_coords
            else:
                print("⚠️  No GPS coordinates found for this activity")
                return []
                
        except Exception as e:
            print(f"❌ Error extracting coordinates: {e}")
            return []
    
    def _get_mock_activities(self):
        """Return mock activity data when Strava API is unavailable."""
        class MockActivity:
            def __init__(self, name, id):
                self.name = name
                self.id = id
                self.type = "Ride"
                
        return [
            MockActivity("Morning Ride through Dutch Countryside", 1),
            MockActivity("Historic Villages Tour", 2)
        ]
    
    def get_mock_locations(self):
        """Return mock Dutch cycling locations."""
        return self.mock_locations

print("✅ Enhanced StravaAPI class defined")

✅ Enhanced StravaAPI class defined


In [8]:
class RouteProcessor:
    """Processes GPS coordinates and extracts location information."""
    
    def __init__(self):
        self.netherlands_cities = [
            "Amsterdam", "Rotterdam", "The Hague", "Utrecht", "Eindhoven", "Tilburg",
            "Groningen", "Almere", "Breda", "Nijmegen", "Enschede", "Haarlem",
            "Arnhem", "Amersfoort", "Zaanstad", "Apeldoorn", "s-Hertogenbosch",
            "Hoofddorp", "Maastricht", "Leiden", "Dordrecht", "Zoetermeer",
            "Zwolle", "Deventer", "Delft", "Alkmaar", "Leeuwarden", "Venlo"
        ]
    
    def coordinates_to_locations(self, coordinates):
        """Convert GPS coordinates to location names."""
        if not coordinates:
            return []
            
        locations = []
        
        for lat, lng in coordinates:
            location = self._approximate_location(lat, lng)
            if location and location not in locations:
                locations.append(location)
        
        return locations[:10]  # Limit to 10 locations
    
    def _approximate_location(self, lat, lng):
        """Approximate location based on coordinates."""
        if not (50.5 <= lat <= 53.8 and 3.0 <= lng <= 7.5):
            return None
            
        lat_regions = {
            (52.0, 52.2): ["The Hague", "Delft", "Leidschendam"],
            (52.2, 52.4): ["Leiden", "Haarlem", "Amsterdam"],
            (52.4, 52.6): ["Amsterdam", "Zaanstad", "Alkmaar"],
            (51.8, 52.0): ["Rotterdam", "Dordrecht", "Breda"],
            (51.4, 51.8): ["Tilburg", "s-Hertogenbosch", "Eindhoven"]
        }
        
        for (min_lat, max_lat), cities in lat_regions.items():
            if min_lat <= lat <= max_lat:
                import random
                return random.choice(cities)
        
        import random
        return random.choice(self.netherlands_cities)
    
    def process_route_data(self, strava_api, activity_id=None):
        """Process route data from Strava or return mock data."""
        if not strava_api.client or not activity_id:
            print("🔄 Using mock location data")
            return strava_api.get_mock_locations()
        
        try:
            activity = strava_api.get_activity_details(activity_id)
            if not activity:
                return strava_api.get_mock_locations()
            
            coordinates = strava_api.extract_route_coordinates(activity)
            if not coordinates:
                return strava_api.get_mock_locations()
            
            locations = self.coordinates_to_locations(coordinates)
            if not locations:
                return strava_api.get_mock_locations()
            
            print(f"✅ Processed {len(locations)} locations from route data")
            return locations
            
        except Exception as e:
            print(f"❌ Error processing route data: {e}")
            return strava_api.get_mock_locations()
    
    def create_route_dataframe(self, locations):
        """Create a DataFrame from location list."""
        return pd.DataFrame({
            "country": ["The Netherlands"] * len(locations),
            "towns": locations
        })

print("✅ RouteProcessor class defined")

✅ RouteProcessor class defined


In [9]:
class StoryGenerator:
    """Generates AI-powered stories about cycling routes using OpenAI."""
    
    def __init__(self, api_key, model="gpt-3.5-turbo"):
        self.model = model
        openai.api_key = api_key
        
    def generate_city_summary(self, city_name):
        """Generate a summary for a single city."""
        prompt = f"Provide a brief historical summary of {city_name}, focusing on origins, importance, major battles or wars, and famous people. Keep it concise and engaging for a cyclist who has visited this place."
        
        try:
            response = openai.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a knowledgeable guide summarizing city histories for cyclists, focusing on salient points and engaging storytelling."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Error generating summary for {city_name}: {str(e)}"
    
    def generate_route_story(self, locations, story_type="golden_age"):
        """Generate a narrative story about a cycling route through multiple locations."""
        if isinstance(locations, list):
            cities = ", ".join(locations)
        else:
            cities = locations
            
        story_prompts = {
            "golden_age": f"""Retell the cyclist's journey as if this was the Dutch Golden Age (17th century), starting and ending at the first location: {cities}.
            Weave a vivid narrative of the route, key historic events, and notable figures tied to these cities.
            Do not simply list the cities - tell the story as if the reader was actually there, experiencing the journey through cobblestone streets, past windmills, and through historic marketplaces.""",
            
            "modern": f"""Tell the story of a modern cyclist's journey through these Dutch locations: {cities}.
            Focus on the contrast between historical significance and contemporary life, mentioning how these places have evolved while maintaining their character.
            Include sensory details about the cycling experience - the feel of the bike paths, the sounds of the city, the architecture that spans centuries.""",
            
            "war_liberation": f"""Narrate a cycling journey through {cities} from the perspective of the liberation of the Netherlands in 1944-1945.
            Focus on the historical significance of these locations during WWII and their liberation, while weaving in the modern cycling experience through these historically significant places."""
        }
        
        prompt = story_prompts.get(story_type, story_prompts["golden_age"])
        
        try:
            response = openai.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a master storyteller who specializes in historical narratives and cycling adventures. Create immersive, engaging stories that transport the reader."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=600,
                temperature=0.3
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Error generating route story: {str(e)}"
    
    def save_story_to_file(self, story, filename, story_type="golden_age"):
        """Save generated story to a file with metadata."""
        story_data = {
            "story": story,
            "story_type": story_type,
            "generated_at": datetime.now().isoformat(),
            "model_used": self.model
        }
        
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                json.dump(story_data, f, indent=2, ensure_ascii=False)
            print(f"📝 Story saved to {filename}")
        except Exception as e:
            print(f"❌ Error saving story: {e}")

print("✅ StoryGenerator class defined")

✅ StoryGenerator class defined


In [10]:
# Initialize all components
strava_api = StravaAPI(strava_client, strava_token_manager)
route_processor = RouteProcessor()
story_generator = StoryGenerator(openai_key, chat_model)

print("🚀 All components initialized successfully!")
print(f"🤖 OpenAI API: {'✅ Ready' if openai_key else '❌ Not configured'}")

# Display connection status
if strava_client:
    print("📍 Data Source: Real Strava activities and GPS routes")
else:
    print("📍 Data Source: Mock Dutch cycling data (for demonstration)")

🚀 All components initialized successfully!
🤖 OpenAI API: ✅ Ready
📍 Data Source: Mock Dutch cycling data (for demonstration)


In [11]:
# Fetch and process route data
print("🔍 Fetching recent cycling activities...")
activities = strava_api.get_recent_activities(limit=5)

# Use the first cycling activity if available
activity_id = None
activity_name = "Mock Activity"

if activities:
    first_activity = activities[0]
    activity_id = getattr(first_activity, 'id', None)
    activity_name = getattr(first_activity, 'name', 'Unknown Activity')
    
    if activity_id:
        print(f"🚴‍♂️ Using activity: {activity_name}")
    else:
        print("🔄 Activity found but no ID available, using mock data")

# Process route data
print("🗺️  Processing route data...")
locations = route_processor.process_route_data(strava_api, activity_id)

# Create DataFrame
route_locations = route_processor.create_route_dataframe(locations)
print(f"📍 Successfully processed {len(locations)} locations for storytelling")

# Display data source info
data_source = "Real Strava GPS data" if (activity_id and strava_client) else "Mock data"
print(f"📊 Data source: {data_source}")

🔍 Fetching recent cycling activities...
🔄 No Strava client available, using mock data
🚴‍♂️ Using activity: Morning Ride through Dutch Countryside
🗺️  Processing route data...
🔄 Using mock location data
📍 Successfully processed 9 locations for storytelling
📊 Data source: Mock data


In [12]:
# Display the route locations
print("📋 Route locations found:")
print(route_locations)

# Extract unique towns for storytelling
all_unique_towns = route_locations["towns"].unique()
joined_string = ", ".join(map(str, all_unique_towns))
print(f"\n🏘️  All unique towns: {textwrap.fill(joined_string, width=80)}")

📋 Route locations found:
           country                towns
0  The Netherlands             Segbroek
1  The Netherlands         Kraayenstein
2  The Netherlands           Kwintsheul
3  The Netherlands          Schipluiden
4  The Netherlands  Berkel en Rodenrijs
5  The Netherlands            Pijnacker
6  The Netherlands         Leidschendam
7  The Netherlands               Marlot
8  The Netherlands           Haagse Bos

🏘️  All unique towns: Segbroek, Kraayenstein, Kwintsheul, Schipluiden, Berkel en Rodenrijs, Pijnacker,
Leidschendam, Marlot, Haagse Bos


In [13]:
# Generate different types of stories
story_types = ["golden_age", "modern", "war_liberation"]

print("📚 Generating stories about your cycling route...\n")

for story_type in story_types:
    print(f"=== {story_type.replace('_', ' ').title()} Story ===")
    story = story_generator.generate_route_story(all_unique_towns, story_type)
    
    # Display the story with proper formatting
    print(textwrap.fill(story, width=80))
    
    # Save story to file
    filename = f"cycling_story_{story_type}.json"
    story_generator.save_story_to_file(story, filename, story_type)
    print(f"\n{'='*50}\n")

📚 Generating stories about your cycling route...

=== Golden Age Story ===
In the midst of the Dutch Golden Age, a lone cyclist set out on a remarkable
journey, his trusty steed gliding effortlessly over the cobblestone streets of
Segbroek. The air was filled with the scent of blooming tulips, a vibrant
tapestry of colors lining the path as he pedaled towards his first destination.
As he passed through Kraayenstein, the cyclist couldn't help but marvel at the
grandeur of the estates that dotted the landscape. These were the homes of
wealthy merchants and nobles, their fortunes built on the bustling trade that
flowed through the nearby port of Rotterdam.  Leaving the opulence behind, the
cyclist ventured on towards Kwintsheul, where the fields stretched out as far as
the eye could see. Here, farmers toiled under the watchful gaze of towering
windmills, their rhythmic turning a testament to the industrious spirit of the
Dutch people.  Next on his journey was Schipluiden, a quaint village

In [14]:
# Token Diagnostics (run this if you have issues)
print("🔧 Strava Token Diagnostics")
print("=" * 40)

if strava_token_manager:
    print("🔍 Testing token validation...")
    is_valid, message = strava_token_manager.validate_token()
    print(f"Token Status: {'✅ Valid' if is_valid else '❌ Invalid'}")
    print(f"Message: {message}")
    
    if not is_valid and strava_token_manager.refresh_token:
        print("\n🔄 Testing token refresh capability...")
        refresh_success, refresh_message = strava_token_manager.refresh_access_token()
        print(f"Refresh Status: {'✅ Success' if refresh_success else '❌ Failed'}")
        print(f"Message: {refresh_message}")
    
    print(f"\n📊 Configuration Status:")
    print(f"   • Client ID: {'✅ Set' if strava_token_manager.client_id else '❌ Missing'}")
    print(f"   • Client Secret: {'✅ Set' if strava_token_manager.client_secret else '❌ Missing'}")
    print(f"   • Access Token: {'✅ Set' if strava_token_manager.access_token else '❌ Missing'}")
    print(f"   • Refresh Token: {'✅ Set' if strava_token_manager.refresh_token else '❌ Missing'}")
    
else:
    print("❌ No token manager available")
    print("💡 Check your .env file configuration")

print(f"\n📋 Quick Fix for Permission Errors:")
print(f"   1. If you get 'activity:read_permission missing', reauthorize your app")
print(f"   2. Use this URL (replace YOUR_CLIENT_ID):")
print(f"   https://www.strava.com/oauth/authorize?client_id=YOUR_CLIENT_ID&response_type=code&redirect_uri=http://localhost&approval_prompt=force&scope=read,activity:read")
print(f"   3. Follow the OAuth flow to get new tokens with proper scopes")

🔧 Strava Token Diagnostics
🔍 Testing token validation...
Token Status: ✅ Valid
Message: Token is valid

📊 Configuration Status:
   • Client ID: ✅ Set
   • Client Secret: ✅ Set
   • Access Token: ✅ Set
   • Refresh Token: ✅ Set

📋 Quick Fix for Permission Errors:
   1. If you get 'activity:read_permission missing', reauthorize your app
   2. Use this URL (replace YOUR_CLIENT_ID):
   https://www.strava.com/oauth/authorize?client_id=YOUR_CLIENT_ID&response_type=code&redirect_uri=http://localhost&approval_prompt=force&scope=read,activity:read
   3. Follow the OAuth flow to get new tokens with proper scopes


In [15]:
# Final Summary
print("✅ Strava Tell Me More notebook complete!")
print("📊 Summary:")
print(f"   • Processed {len(all_unique_towns)} unique locations")
print(f"   • Generated {len(story_types)} different story types")
print(f"   • Data source: {data_source}")
print(f"   • Stories saved as JSON files in current directory")

if not strava_client:
    print("\n💡 To use real Strava data:")
    print("   1. Configure your .env file with proper tokens")
    print("   2. Ensure your app has 'activity:read' scope")
    print("   3. Run the diagnostics cell above for troubleshooting")

✅ Strava Tell Me More notebook complete!
📊 Summary:
   • Processed 9 unique locations
   • Generated 3 different story types
   • Data source: Mock data
   • Stories saved as JSON files in current directory

💡 To use real Strava data:
   1. Configure your .env file with proper tokens
   2. Ensure your app has 'activity:read' scope
   3. Run the diagnostics cell above for troubleshooting
